<a href="https://colab.research.google.com/github/yisuznazi/juegosAjedrez/blob/main/ajedrez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


20.juegos de ajedrez


Distribución de ELO de jugadores blancos vs negros
¿Quién gana más, blancas o negras?
Partidas rated vs no rated

Análisis de aperturas columna eco y opening_name

¿Qué aperturas tienen mayor tasa de victoria?
¿Las aperturas más largas opening_ply favorecen a alguien?

Correlaciones interesantes

¿Más turnos = partidas más igualadas en ELO?
¿Diferencia de ELO predice el resultado?
¿El tiempo de control increment_code afecta quién gana?

Minería de texto columna moves

Movimientos más frecuentes en las primeras jugadas
Patrones de jugadas ganadoras

Clasificación / ML opcional

Predecir el ganador con white_elo, black_elo, opening_name, turns



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [5]:
df=pd.read_csv('games.csv')

In [6]:
df.head()

,id,rated,created_at,last_move_at,turns,victory_status,winner,increment_code,white_id,white_rating,black_id,black_rating,moves,opening_eco,opening_name,opening_ply
0,TZJHLljE,False,1.504210e+12,1.504210e+12,13,outoftime,white,15+2,bourgris,1500,a-00,1191,d4 d5 c4 c6 cxd5 e6 dxe6 fxe6 Nf3 Bb4+ Nc3 Ba5...,D10,Slav Defense: Exchange Variation,5
1,l1NXvwaE,True,1.504130e+12,1.504130e+12,16,resign,black,5+10,a-00,1322,skinnerua,1261,d4 Nc6 e4 e5 f4 f6 dxe5 fxe5 fxe5 Nxe5 Qd4 Nc6...,B00,Nimzowitsch Defense: Kennedy Variation,4
2,mIICvQHh,True,1.504130e+12,1.504130e+12,61,mate,white,5+10,ischia,1496,a-00,1500,e4 e5 d3 d6 Be3 c6 Be2 b5 Nd2 a5 a4 c5 axb5 Nc...,C20,King's Pawn Game: Leonardis Variation,3
3,kWKvrqYL,True,1.504110e+12,1.504110e+12,61,mate,white,20+0,daniamurashov,1439,adivanov2009,1454,d4 d5 Nf3 Bf5 Nc3 Nf6 Bf4 Ng4 e3 Nc6 Be2 Qd7 O...,D02,Queen's Pawn Game: Zukertort Variation,3
4,9tXo1AUZ,True,1.504030e+12,1.504030e+12,95,mate,white,30+3,nik221107,1523,adivanov2009,1469,e4 e5 Nf3 d6 d4 Nc6 d5 Nb4 a3 Na6 Nc3 Be7 b4 N...,C41,Philidor Defense,5


In [7]:
df.tail(
)

,id,rated,created_at,last_move_at,turns,victory_status,winner,increment_code,white_id,white_rating,black_id,black_rating,moves,opening_eco,opening_name,opening_ply
20053,EfqH7VVH,True,1.499791e+12,1.499791e+12,24,resign,white,10+10,belcolt,1691,jamboger,1220,d4 f5 e3 e6 Nf3 Nf6 Nc3 b6 Be2 Bb7 O-O Be7 Ne5...,A80,Dutch Defense,2
20054,WSJDhbPl,True,1.499698e+12,1.499699e+12,82,mate,black,10+0,jamboger,1233,farrukhasomiddinov,1196,d4 d6 Bf4 e5 Bg3 Nf6 e3 exd4 exd4 d5 c3 Bd6 Bd...,A41,Queen's Pawn,2
20055,yrAas0Kj,True,1.499698e+12,1.499698e+12,35,mate,white,10+0,jamboger,1219,schaaksmurf3,1286,d4 d5 Bf4 Nc6 e3 Nf6 c3 e6 Nf3 Be7 Bd3 O-O Nbd...,D00,Queen's Pawn Game: Mason Attack,3
20056,b0v4tRyF,True,1.499696e+12,1.499697e+12,109,resign,white,10+0,marcodisogno,1360,jamboger,1227,e4 d6 d4 Nf6 e5 dxe5 dxe5 Qxd1+ Kxd1 Nd5 c4 Nb...,B07,Pirc Defense,4
20057,N8G2JHGG,True,1.499643e+12,1.499644e+12,78,mate,black,10+0,jamboger,1235,ffbob,1339,d4 d5 Bf4 Na6 e3 e6 c3 Nf6 Nf3 Bd7 Nbd2 b5 Bd3...,D00,Queen's Pawn Game: Mason Attack,3


In [9]:
df.dtypes

,0
id,object
rated,bool
created_at,float64
last_move_at,float64
turns,int64
victory_status,object
winner,object
increment_code,object
white_id,object
white_rating,int64


In [10]:
df.shape

(20058, 16)

In [11]:
df.isnull().sum()

,0
id,0
rated,0
created_at,0
last_move_at,0
turns,0
victory_status,0
winner,0
increment_code,0
white_id,0
white_rating,0


In [12]:
df.columns


Index(['id', 'rated', 'created_at', 'last_move_at', 'turns', 'victory_status',
       'winner', 'increment_code', 'white_id', 'white_rating', 'black_id',
       'black_rating', 'moves', 'opening_eco', 'opening_name', 'opening_ply'],
      dtype='object')

In [16]:
# Cuántas partidas gano cada uno agrupado por turnos
ganador_por_turnos = df.groupby(['winner', 'turns']).size().reset_index(name='cantidad_partidas')
print(ganador_por_turnos)

    winner  turns  cantidad_partidas
0    black      1                  2
1    black      2                108
2    black      3                 13
3    black      4                 36
4    black      5                  3
5    black      6                 26
6    black      7                  4
7    black      8                 32
8    black      9                  8
9    black     10                 44
10   black     11                 10
11   black     12                 56
12   black     13                 14
13   black     14                 71
14   black     15                 10
15   black     16                 81
16   black     17                 15
17   black     18                 84
18   black     19                 18
19   black     20                104
20   black     21                 21
21   black     22                 97
22   black     23                 20
23   black     24                114
24   black     25                 25
25   black     26                135
2